In [2]:
"""
FINDINGS FROM DATA EXPLORATION:
1. Data set contains 6362620 rows and 11 columns, that is, 6362620 transactions are observed in terms of 11 quantities
2. There are no null and duplicate values in dataset
3. Only 0.13 % transactions are  fraud among total transactions
4. Transactions are of different types: cash-in, cash-out, debit, transfer, payment and only cashout and transfer contain frauds
5. Fraud transactions usually involve higher amounts.
6. newbalanceOrig frequently becomes zero in fraud cases.
7. Dataset is highly imbalanced.
"""

'\nFINDINGS FROM DATA EXPLORATION:\n1. Data set contains 6362620 rows and 11 columns, that is, 6362620 transactions are observed in terms of 11 quantities\n2. There are no null and duplicate values in dataset\n3. Only 0.13 % transactions are  fraud among total transactions\n4. Transactions are of different types: cash-in, cash-out, debit, transfer, payment and only cashout and transfer contain frauds\n5. Fraud transactions usually involve higher amounts.\n6. newbalanceOrig frequently becomes zero in fraud cases.\n7. Dataset is highly imbalanced.\n'

In [3]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/data.csv")

In [4]:
# 11 coulmns and 6362620 rows
df.shape

(6362620, 11)

In [5]:
#mean, std dev, min, max, med important factors
df.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,2.514687e-06
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.585775e-03
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,0.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+00


In [6]:
# data type of each column
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [7]:
# this shows no null value is present in any column
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [8]:
# this shows no duplicate value is present in any column
df.duplicated().sum()

np.int64(0)

In [9]:
# Normal Transaction == 0
# Fraud Transaction == 1 
# Normalzse give fraction like fraud/total
# this shows on;y 0.13 % transactions are fraud
df["isFraud"].value_counts(normalize=True) * 100

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [10]:
# this shows how many transactions are fraud or not for each type
pd.crosstab(
    df["type"],
    df["isFraud"]
)

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [11]:
# description of fraud transactions
fraud_df = df[df["isFraud"] == 1]
fraud_df["type"].value_counts()
fraud_df["amount"].describe()


count    8.213000e+03
mean     1.467967e+06
std      2.404253e+06
min      0.000000e+00
25%      1.270913e+05
50%      4.414234e+05
75%      1.517771e+06
max      1.000000e+07
Name: amount, dtype: float64

In [12]:
# description of normal transactions
normal_df = df[df["isFraud"] == 0]
normal_df["amount"].describe()

count    6.354407e+06
mean     1.781970e+05
std      5.962370e+05
min      1.000000e-02
25%      1.336840e+04
50%      7.468472e+04
75%      2.083648e+05
max      9.244552e+07
Name: amount, dtype: float64

In [ ]:
"""
describes relation of fraud balance with these columns:
oldbalanceOrg == old balance of origin/sender
newbalanceOrg == new balance of origin/sender
oldbalanceDest == old balance of destination/receiver
newbalanceDest == new balance of destination/receiver
"""
fraud_df[
    [
        "oldbalanceOrg",
        "newbalanceOrig",
        "oldbalanceDest",
        "newbalanceDest"
    ]
].describe()

,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
count,8.213000e+03,8.213000e+03,8.213000e+03,8.213000e+03
mean,1.649668e+06,1.923926e+05,5.442496e+05,1.279708e+06
std,3.547719e+06,1.965666e+06,3.336421e+06,3.908817e+06
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.258224e+05,0.000000e+00,0.000000e+00,0.000000e+00
50%,4.389835e+05,0.000000e+00,0.000000e+00,4.676420e+03
75%,1.517771e+06,0.000000e+00,1.478287e+05,1.058725e+06
max,5.958504e+07,4.958504e+07,2.362305e+08,2.367265e+08
